# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities in the dataset (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"  Record Set Name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    Description: {getattr(rs, 'description', 'No description')}\n")

    # List fields in each record set
    print(f"    Fields:")
    for field in rs.fields:
        print(f"      Field Name: {field.name}")
        print(f"        @id: {field.id}")
        print(f"        DataType: {getattr(field, 'data_type', 'unknown')}")
        print(f"        Description: {getattr(field, 'description', 'N/A')}\n")
    print('-'*60)

# Example: Iterate records from the first record set
if len(record_sets) > 0:
    example_record_set_id = record_sets[0].id
    print(f"\nFirst 2 records of Record Set @id: {example_record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load the data from each record set into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Record Set @id: {rs_id} --> DataFrame Columns: {df.columns.tolist()}")
    print(df.head(), '\n')

# Select a record set for further analysis
if record_set_ids:
    analysis_record_set_id = record_set_ids[0]
    print(f"Selected Record Set @id for EDA: {analysis_record_set_id}")
    print(f"Columns: {dataframes[analysis_record_set_id].columns.tolist()}")
    dataframes[analysis_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply processing: filter by threshold, normalize numeric fields, categorize/group data.

All fields are referenced by `@id`.

In [ ]:
# Choose a numeric field @id and a group field @id from the selected record set
# For demonstration, pick fields with 'age' and 'sex' in their names (replace as necessary based on schema)
df = dataframes[analysis_record_set_id]

# Find numeric and categorical fields by inspecting columns
numeric_field_candidates = [col for col in df.columns if 'age' in col.lower() or df[col].dtype in ['int64', 'float64']]
numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]

group_field_candidates = [col for col in df.columns if 'sex' in col.lower() or 'msi' in col.lower() or df[col].dtype == 'object']
group_field_id = group_field_candidates[0] if group_field_candidates else df.columns[0]

# Filter records by numeric threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by chosen group field if it exists
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships between fields.

In [ ]:
# Plot numeric field distribution
plt.figure(figsize=(8, 4))
plt.hist(df[numeric_field_id].dropna(), bins=10, color='skyblue', edgecolor='black')
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Plot group field vs numeric field (boxplot)
if group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrates loading, processing, and visualizing tabular clinical and molecular data using `mlcroissant`. Key findings, such as the distribution of numeric variables (e.g., age) and relationships to molecular or clinical categories (e.g., MSI status or sex), can be explored further.

For advanced modeling and analytics, reference column and field `@id`s throughout for reproducible, schema-compliant processing.